# Daily Challenge : Construire un système RAG (Retrieval Augmented Generation)

Ce notebook construit un système RAG complet avec **Langchain** et **Hugging Face**, en suivant la consigne du "Daily Challenge".

## Étape 1 : Configurer l'environnement

Cette étape installe toutes les bibliothèques nécessaires à la construction du système RAG. Chaque bibliothèque a un rôle précis :
- **langchain** (+ `langchain-core`, `langchain-community`, `langchain-text-splitters`, `langchain-huggingface`, `langchain-classic`) : orchestre tous les composants du RAG (chargement, découpage, recherche, génération) ;
- **transformers** : fournit les modèles pré-entraînés (embeddings et LLM) ;
- **sentence-transformers** : génère les embeddings (vecteurs numériques) à partir du texte ;
- **datasets** : charge le jeu de données Hugging Face ;
- **faiss-cpu** : la base de données vectorielle qui permet une recherche rapide par similarité ;
- **torch** : le moteur de calcul (PyTorch) utilisé en arrière-plan par les modèles.


In [1]:
# 1. On désinstalle d'abord torchvision et torchaudio : ils ne servent à rien dans ce projet
#    (texte uniquement, pas d'images) et leur version préinstallée sur Colab entre en conflit
#    avec la version de torch qu'on installe juste après.
!pip uninstall -y -q torchvision torchaudio

# 2. On installe (ou met à jour) toutes les bibliothèques nécessaires au projet
!pip install -q -U torch transformers sentence-transformers datasets faiss-cpu
!pip install -q -U langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface langchain-classic

print("Installation terminée.")
print(" Redémarre maintenant la session Colab (Exécution > Redémarrer la session), puis relance les cellules à partir d'ici.")


Installation terminée.
 Redémarre maintenant la session Colab (Exécution > Redémarrer la session), puis relance les cellules à partir d'ici.


## Étape 2 : Charger le jeu de données

On charge le jeu de données `databricks/databricks-dolly-15k` avec `HuggingFaceDatasetLoader`, qui transforme directement chaque ligne du dataset en `Document` Langchain (prêt à être utilisé par la suite).

D'après la consigne, on utilise la colonne `context` comme contenu principal de chaque document.



In [2]:
# On s'assure que la librairie datasets est à jour
!pip install -Uq datasets

# HuggingFaceDatasetLoader se trouve maintenant dans langchain_community.document_loaders
from langchain_community.document_loaders import HuggingFaceDatasetLoader

# Nom du jeu de données sur le Hugging Face Hub
dataset_name = "databricks/databricks-dolly-15k"
# Nom de la colonne utilisée comme contenu principal du document (imposé par la consigne)
page_content_column = "context"

# On crée le loader, puis on charge les données : chaque ligne devient un objet Document Langchain
loader = HuggingFaceDatasetLoader(dataset_name, page_content_column)
data = loader.load()

# On affiche les deux premiers documents pour vérifier que le chargement a fonctionné
print("Nombre de documents chargés :", len(data))
print(data[:2])


/tmp/ipykernel_28308/2295441178.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import HuggingFaceDatasetLoader


Nombre de documents chargés : 15011
[Document(metadata={'instruction': 'When did Virgin Australia start operating?', 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}, page_content='"Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia\'s domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney."'), Document(metadata={'instruction': 'Which is a species of fish? Tope or Rope', 'response': 'Tope', 'category': 'classification'}, page_content='""')]


## Étape 3 : Découper les documents en chunks

Les modèles de langage ont une limite de texte qu'ils peuvent traiter en une seule fois. On découpe donc les documents en petits morceaux (chunks) qui se chevauchent légèrement, pour ne jamais couper une idée importante en deux.

- `chunk_size = 1000` : taille maximale d'un chunk, en nombre de caractères ;
- `chunk_overlap = 150` : nombre de caractères communs entre deux chunks consécutifs (la "zone de chevauchement").

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# On crée le découpeur avec les réglages imposés par la consigne
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

# On découpe tous les documents chargés à l'étape précédente
docs = text_splitter.split_documents(data)

# On affiche le nombre total de chunks obtenus, et le premier chunk pour vérifier le résultat
print("Nombre de chunks obtenus :", len(docs))
print(docs[0])


Nombre de chunks obtenus : 18502
page_content='"Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney."' metadata={'instruction': 'When did Virgin Australia start operating?', 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


## Étape 4 : Créer les embeddings du texte

Un embedding est une représentation numérique (un vecteur) d'un texte, qui capture son sens. Deux textes proches en signification auront des vecteurs proches. On utilise ici un modèle `sentence-transformers`, spécialisé dans ce type de représentation.



In [4]:
# HuggingFaceEmbeddings a été déplacé dans le package langchain_huggingface
from langchain_huggingface import HuggingFaceEmbeddings

# Nom exact du modèle d'embedding sur le Hugging Face Hub (attention à la casse : "L6", pas "l6")
modelPath = "sentence-transformers/all-MiniLM-L6-v2"

# model_kwargs : options passées au modèle (ici, on force l'exécution sur le processeur, pas le GPU)
model_kwargs = {'device': 'cpu'}
# encode_kwargs : options d'encodage (ici, on ne normalise pas les vecteurs obtenus)
encode_kwargs = {'normalize_embeddings': False}

# On initialise l'objet embeddings : c'est lui qui transformera chaque chunk de texte en vecteur
embeddings = HuggingFaceEmbeddings(
    model_name=modelPath,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

# Test optionnel : on vérifie que la création d'un embedding fonctionne sur une phrase simple
text = "This is a test document."
query_result = embeddings.embed_query(text)
print("Taille du vecteur obtenu :", len(query_result))
print("3 premières valeurs du vecteur :", query_result[:3])


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Taille du vecteur obtenu : 384
3 premières valeurs du vecteur : [-0.03833852335810661, 0.12346465140581131, -0.02864294871687889]


## Étape 5 : Créer le magasin vectoriel (vector store)

FAISS indexe tous les embeddings des chunks, ce qui permet de retrouver très rapidement les chunks les plus proches (donc les plus pertinents) d'une question donnée.



In [5]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

# 1. On calcule la dimension des vecteurs produits par notre modèle d'embedding
#    (nécessaire pour créer un index FAISS de la bonne taille)
embedding_dim = len(embeddings.embed_query("hello world"))

# 2. On crée un index FAISS vide.
#    IndexFlatL2 = recherche exacte basée sur la distance euclidienne (L2) entre les vecteurs.
index = faiss.IndexFlatL2(embedding_dim)

# 3. On crée le vector store FAISS : au départ, il est vide (aucun document indexé)
#    - embedding_function : le modèle qui transforme le texte en vecteurs
#    - index : l'index FAISS créé ci-dessus
#    - docstore : l'endroit où sont stockés les documents eux-mêmes (ici, en mémoire)
#    - index_to_docstore_id : dictionnaire vide au départ, rempli automatiquement par add_documents
db = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

# 4. On ajoute nos chunks (docs) au vector store avec add_documents (interface unifiée Langchain)
#    On donne un identifiant unique (UUID) à chaque chunk
from uuid import uuid4
ids = [str(uuid4()) for _ in range(len(docs))]
db.add_documents(documents=docs, ids=ids)

print("Vector store FAISS créé avec succès.")
print("Nombre de chunks indexés :", db.index.ntotal)

# 5. Petite vérification de l'interface unifiée : on teste similarity_search directement,
#    sans passer par un retriever, pour voir les documents les plus proches d'une phrase simple.
apercu = db.similarity_search("What is cheesemaking?", k=2)
print("\nAperçu avec similarity_search (2 chunks les plus proches) :")
for d in apercu:
    print("-", d.page_content[:150].replace(chr(10), " "), "...")


Vector store FAISS créé avec succès.
Nombre de chunks indexés : 18502

Aperçu avec similarity_search (2 chunks les plus proches) :
- "The goal of cheese making is to control the spoiling of milk into cheese. The milk is traditionally from a cow, goat, sheep or buffalo, although, in  ...
- cut using long, blunt knives and 'blocked' (stacked, cut and turned) by the cheesemaker to promote the release of cheese whey in a process known as 'c ...


## Étape 6 : Préparer le modèle de langage (LLM)

Ici, le "LLM" est en réalité un modèle de **question-réponse extractive** (`AutoModelForQuestionAnswering`) : contrairement à un modèle génératif (comme GPT), il ne "rédige" pas une réponse — il **repère directement un passage exact** dans le texte fourni (le "contexte") qui répond à la question.

In [6]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
import torch

# Nom du modèle de question-réponse imposé par la consigne
model_name = "Intel/dynamic_tinybert"

# On charge le tokenizer (qui transforme le texte en nombres compréhensibles par le modèle)
# et le modèle de question-réponse lui-même.
tokenizer = AutoTokenizer.from_pretrained(model_name, padding=True, truncation=True, max_length=512)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)
model.eval()  # mode "évaluation" : on n'entraîne pas le modèle, on l'utilise juste pour prédire

# On essaie de créer le pipeline Hugging Face "question-answering", comme demandé par la consigne.
qa_pipeline = None
try:
    qa_pipeline = pipeline(
        "question-answering",
        model=model,
        tokenizer=tokenizer,
        return_tensors='pt'
    )
    print(" Pipeline `question-answering` créé avec succès.")
except KeyError as e:
    print(" La tâche 'question-answering' n'est pas disponible dans cette version de transformers.")
    print("   On utilisera le modèle directement (fonction manuelle ci-dessous) à la place.")
    print("   Détail :", e)


def repondre_qa_extractive(question, context, max_length=512):
    """
    Fonction manuelle qui reproduit ce que fait pipeline("question-answering") en interne :
    1. on donne la question ET le contexte au modèle (les deux ensemble, séparés par un token spécial)
    2. le modèle prédit la position de DÉBUT et de FIN de la réponse à l'intérieur du contexte
       (deux listes de scores : start_logits et end_logits, une valeur par mot/token)
    3. on prend la position où le score est le plus élevé (argmax) pour le début et pour la fin
    4. on découpe et on décode le texte correspondant à cette position : c'est la réponse
    """
    inputs = tokenizer(question, context, return_tensors="pt", truncation=True, max_length=max_length)
    with torch.no_grad():  # pas besoin de calculer les gradients, on ne fait que prédire
        outputs = model(**inputs)

    start_idx = torch.argmax(outputs.start_logits)
    end_idx = torch.argmax(outputs.end_logits) + 1
    if end_idx <= start_idx:  # sécurité : évite une réponse vide ou incohérente
        end_idx = start_idx + 1

    answer_tokens = inputs["input_ids"][0][start_idx:end_idx]
    answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)
    score = torch.softmax(outputs.start_logits, dim=-1)[0, start_idx].item()
    return answer, score


Invalid model-index. Not loading eval results into CardData.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: Intel/dynamic_tinybert
Key              | Status     |  | 
-----------------+------------+--+-
fit_dense.weight | UNEXPECTED |  | 
fit_dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 La tâche 'question-answering' n'est pas disponible dans cette version de transformers.
   On utilisera le modèle directement (fonction manuelle ci-dessous) à la place.
   Détail : "Unknown task question-answering, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"


## Étape 7 : Construire la chaîne Retrieval QA

La chaîne `RetrievalQA` relie deux éléments :
- le **retriever** : qui va chercher les chunks les plus pertinents dans la base FAISS ;
- le **LLM** : qui génère une réponse à partir de ces chunks.

`chain_type="refine"` (imposé par la consigne) signifie que la chaîne lit les chunks **un par un**, et **affine** sa réponse à chaque nouveau chunk lu — contrairement à `"stuff"`, qui empile tous les chunks d'un coup dans le prompt.



In [7]:
# RetrievalQA a changé d'emplacement plusieurs fois dans Langchain ; dans les versions récentes,
# il se trouve dans langchain_classic.chains.
from langchain_classic.chains import RetrievalQA
# HuggingFacePipeline a été déplacé dans le package langchain_huggingface
from langchain_huggingface import HuggingFacePipeline

# Le retriever va chercher les k chunks les plus pertinents pour une question donnée
retriever = db.as_retriever(search_kwargs={"k": 4})  # k réglable : plus de chunks = plus de contexte, mais parfois moins précis

use_fallback = True  # par défaut, on prévoit d'utiliser la solution de repli (fonction manuelle)
qa = None

if qa_pipeline is not None:
    # Le pipeline officiel existe : on tente de construire la chaîne RAG comme demandé dans la consigne
    try:
        llm = HuggingFacePipeline(
            pipeline=qa_pipeline,
            model_kwargs={"temperature": 0.7, "max_length": 512},
        )
        qa = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type="refine",
            retriever=retriever,
            return_source_documents=False
        )
        use_fallback = False
        print(" Chaîne RetrievalQA créée avec succès.")
    except Exception as e:
        print(" Impossible de créer la chaîne RetrievalQA avec ce pipeline (erreur ci-dessous).")
        print("   On utilisera la solution de repli (fonction manuelle) à l'étape 8.")
        print("   Détail de l'erreur :", e)
else:
    print("ℹ Aucun pipeline `question-answering` disponible : on utilisera directement la fonction")
    print("   manuelle définie à l'étape 6 pour répondre aux questions à l'étape 8.")


ℹ Aucun pipeline `question-answering` disponible : on utilisera directement la fonction
   manuelle définie à l'étape 6 pour répondre aux questions à l'étape 8.


## Étape 8 : Tester le système RAG

On pose la question `"What is cheesemaking?"` (imposée par la consigne) et on vérifie que le système :
1. retrouve bien des chunks pertinents dans le jeu de données ;
2. génère une réponse cohérente à partir de ces chunks.



In [8]:
# Question de test imposée par la consigne
question = "What is cheesemaking?"

if not use_fallback:
    # Cas normal : on utilise la chaîne RetrievalQA construite à l'étape 7
    result = qa.invoke({"query": question})
    print("Réponse :", result["result"])
else:
    # Solution de repli : on reproduit à la main ce que fait RetrievalQA en interne.
    # 1. On récupère les chunks les plus pertinents avec le retriever
    relevant_docs = retriever.invoke(question)

    # 2. On assemble leur contenu pour former le "contexte" donné au modèle de question-réponse
    context = " ".join(doc.page_content for doc in relevant_docs)

    # 3. On appelle notre fonction manuelle définie à l'étape 6 (repondre_qa_extractive)
    answer, score = repondre_qa_extractive(question, context)

    print("Réponse :", answer)
    print("Score de confiance du modèle :", round(score, 3))


Réponse : to control the spoiling of milk into cheese
Score de confiance du modèle : 0.665
